# DD2424 - Assignment 2
This is the main notebook containing the second assignment of course DD2424.

Note that the dataset is loaded from folder in assignment1!

In [1]:
# needed imports
import matplotlib.pyplot as plt
import numpy as np

from model import Model
from nodes import CrossEntropyLoss
from scaler import Scaler
from utils import load_batch, calculate_mean_grad_difference
from torch_gradient_computations import ComputeGradsWithTorch


## 1. Test gradient computation

In [2]:
X, Y, y = load_batch("data_batch_1")
scaler = Scaler()
X_train = scaler.fit_transform(X)

/home/josef/Uni/P4/deep_learning/DD2424-assignments/assignment2/src/utils.py:20: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict = pickle.load(fo, encoding='bytes')


In [3]:
model = Model(32*32*3, 100, 10)
loss_node = CrossEntropyLoss()

Y_pred = model.forward(X_train[:,:100])
loss = loss_node.forward(Y_pred, Y[:, :100])
print(f"loss computed {loss}")

loss computed 3.1049067420443452


In [4]:
# calculate my gradient
grad = loss_node.backward()
print(grad.shape)
model.backward(grad)

# calculate torch gradient
grads_torch = ComputeGradsWithTorch(X_train[:, :100], y[:100], model.layers[0].W, model.layers[0].b, model.layers[2].W, model.layers[2].b)

(10, 100)
Loss computed with PyTorch: 3.104906742044


In [5]:
print(model.layers[2].grad_b)

[[-0.0304274 ]
 [-0.08518595]
 [ 0.01107502]
 [ 0.00499653]
 [-0.06670064]
 [-0.02543581]
 [ 0.00066563]
 [-0.03376153]
 [ 0.27279882]
 [-0.04802467]]


In [6]:
print(grads_torch["b2"])

[[-0.0304274 ]
 [-0.08518595]
 [ 0.01107502]
 [ 0.00499653]
 [-0.06670064]
 [-0.02543581]
 [ 0.00066563]
 [-0.03376153]
 [ 0.27279882]
 [-0.04802467]]


In [7]:
# compare the gradients
W1_diff = calculate_mean_grad_difference(model.layers[0].grad_W.flatten(), grads_torch["W1"].flatten())
W2_diff = calculate_mean_grad_difference(model.layers[2].grad_W.flatten(), grads_torch["W2"].flatten())
b1_diff = calculate_mean_grad_difference(model.layers[0].grad_b.flatten(), grads_torch["b1"].flatten())
b2_diff = calculate_mean_grad_difference(model.layers[2].grad_b.flatten(), grads_torch["b2"].flatten())

print("First layer gradients:")
print(f"Mean absolute difference in W1 gradients: {W1_diff}")
print(f"Mean absolute difference in b1 gradients: {b1_diff}")
print("Second layer gradients:")
print(f"Mean absolute difference in W2 gradients: {W2_diff}")
print(f"Mean absolute difference in b2 gradients: {b2_diff}")

First layer gradients:
Mean absolute difference in W1 gradients: 9.538691045058804e-17
Mean absolute difference in b1 gradients: 7.536338501576082e-17
Second layer gradients:
Mean absolute difference in W2 gradients: 5.078180117598879e-17
Mean absolute difference in b2 gradients: 9.829626290928924e-17


### Notes:
- The gradients are almost identical!